In [2]:
pip install tensorflow

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/351.2 MB 3.6 MB/s eta 0:01:37
   ---------------------------------------- 1.6/351.2 MB 3.8 MB/s eta 0:01:32
   ---------------------------------------- 2.1/351.2 MB 3.6 MB/s eta 0:01:38
   ---------------------------------------- 2.9/351.2 MB 3.5 MB/s eta 0:01:39
   ---------------------------------------- 3.4/351.2 MB 3.5 MB/s eta 0:01:39
   ---------------------------------------- 4.2/351.2 MB 3.5 MB/s eta 0:01:41
    --------------------------------------- 5.0/351.2 MB 3.5 MB/s eta 0:01:39
    --------------------------------------- 5.8/351.2 MB 3.5 MB/s eta 0:01:39
    --------------------------------------- 6.8/351.2 MB 3.6 MB/s eta 0:01:37
    ------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import shutil
import sys
from pathlib import Path
import tensorflow as tf


In [4]:
# --- CONFIGURATION ---
DATASET_URL = "http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz"

# --- AUTO-DETECT ENVIRONMENT ---
# Automatically sets the correct path whether you are in Colab or Local Windows
if 'google.colab' in sys.modules:
    print("🌐 Google Colab environment detected!")
    
    # Connect (mount) to Google Drive
    from google.colab import drive
    print("🔗 Mounting Google Drive...")
    drive.mount('/content/drive')
    
    # Save directly to your Google Drive so the data persists
    BASE_PATH = "/content/drive/MyDrive/Food101_Data"
else:
    print("🖥️ Local environment detected!")
    BASE_PATH = r"D:\Food101_Data"

base_dir = Path(BASE_PATH)
FINAL_DIR = base_dir / "dataset"

🖥️ Local environment detected!


In [5]:

def setup_dataset():
    print("🍽️ Starting Food-101 Dataset Setup...")
    os.makedirs(base_dir, exist_ok=True)

    print("\n📥 Checking for dataset archive...")
    
    try:
        dataset_path = tf.keras.utils.get_file(
            fname="food-101",
            origin=DATASET_URL,
            untar=True,
            cache_dir=str(base_dir),
            cache_subdir="raw_data"
        )
    except Exception as e:
        print(f"❌ Download or extraction failed: {e}")
        return

    print("✅ Download and extraction verified!")

    # 3. Create our clean Train/Test directories
    train_dir = FINAL_DIR / "train"
    test_dir = FINAL_DIR / "test"
    
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    print("\n🔍 Locating extracted files...")
    
    # --- SMART PATH FINDER (Fixes the nested folder issue) ---
    raw_data_dir = base_dir / "raw_data"
    EXTRACT_FOLDER = None
    
    # Search for the "meta" folder dynamically
    for path in raw_data_dir.rglob("meta"):
        if (path / "train.txt").exists():
            EXTRACT_FOLDER = path.parent
            break

    if EXTRACT_FOLDER is None:
        print(f"❌ CRITICAL ERROR: Could not find 'meta/train.txt' anywhere inside {raw_data_dir}.")
        print("This usually means the extraction silently failed (common on Google Drive due to limits).")
        print("Try deleting the 'raw_data' folder and running the script again.")
        return

    meta_dir = EXTRACT_FOLDER / "meta"
    images_dir = EXTRACT_FOLDER / "images"
    print(f"✅ Found data at: {EXTRACT_FOLDER}")

    # 4. Read the meta files and organize the images
    print("\n📁 Reorganizing images into clean Train and Test folders...")
    print("⏳ Note: Moving 100,000 files on Google Drive can take a while. Please do not close the tab.")

    for split in ['train', 'test']:
        txt_file = meta_dir / f"{split}.txt"
        target_dir = train_dir if split == 'train' else test_dir

        with open(txt_file, 'r') as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            img_path_no_ext = line.strip()
            class_name = img_path_no_ext.split('/')[0]
            
            # Create the class folder
            class_dir = target_dir / class_name
            os.makedirs(class_dir, exist_ok=True)

            # Move the file
            src_file = images_dir / f"{img_path_no_ext}.jpg"
            dst_file = class_dir / f"{img_path_no_ext.split('/')[1]}.jpg"

            if src_file.exists():
                shutil.move(str(src_file), str(dst_file))

            # Print progress every 2500 images
            if i % 2500 == 0:
                print(f"   Organized {i} {split} images...")

    print("\n✅ Dataset organization complete! Your data is ready for training.")
    print(f"🧹 You can now safely delete the '{base_dir / 'raw_data'}' folder to free up 5GB of storage space.")


In [6]:
if __name__ == "__main__":
    setup_dataset()

🍽️ Starting Food-101 Dataset Setup...

📥 Checking for dataset archive...
4996278331/4996278331 ━━━━━━━━━━━━━━━━━━━━ 967s 0us/step
✅ Download and extraction verified!

🔍 Locating extracted files...
✅ Found data at: D:\Food101_Data\raw_data\food-101\food-101

📁 Reorganizing images into clean Train and Test folders...
⏳ Note: Moving 100,000 files on Google Drive can take a while. Please do not close the tab.
   Organized 0 train images...
   Organized 2500 train images...
   Organized 5000 train images...
   Organized 7500 train images...
   Organized 10000 train images...
   Organized 12500 train images...
   Organized 15000 train images...
   Organized 17500 train images...
   Organized 20000 train images...
   Organized 22500 train images...
   Organized 25000 train images...
   Organized 27500 train images...
   Organized 30000 train images...
   Organized 32500 train images...
   Organized 35000 train images...
   Organized 37500 train images...
   Organized 40000 train images...
   

In [7]:
import os

if os.path.exists('/content/local_food101/dataset'):
    print("✅ The folder is definitely there! The Colab UI just needs to be refreshed.")
else:
    print("❌ The folder is gone. Your Colab runtime disconnected and wiped the temporary memory.")

❌ The folder is gone. Your Colab runtime disconnected and wiped the temporary memory.
